# Friction Surface Computation & Visualization

Interactive notebook for building friction rasters, computing least-cost paths,
and visualizing the results for the Alaska fuel delivery project.

**Runs on Jetstream2** alongside the existing pipeline notebooks.

### Visualizations included:
1. Individual input layer maps (LULC, slope, permafrost, roads, rivers, DEM)
2. Composite friction raster maps (road, barge, sky)
3. Facility overlay on friction surfaces
4. Least-cost path traces
5. Haversine vs friction path comparison
6. Seasonal friction heatmaps
7. Regional friction summary bar charts
8. Cost distribution histograms

## 1. Setup & Imports

In [ ]:
import numpy as np
import rasterio
from rasterio.plot import show
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
import geopandas as gpd
import pandas as pd
import duckdb
import os
import sys

# Add parent directory to path for project imports
sys.path.insert(0, '..')

import friction_config
import friction_surface
import pipeline

%matplotlib inline
plt.rcParams['figure.figsize'] = (16, 10)
plt.rcParams['figure.dpi'] = 100

print("Imports loaded successfully.")

## 2. Load GEE Rasters

In [ ]:
# Load all GEE-exported rasters
rasters = friction_surface.load_rasters()

# Print summary
print(f"\n{'Layer':<20} {'Shape':<20} {'Dtype':<10} {'Min':<10} {'Max':<10}")
print("-" * 70)
for name, (data, profile) in rasters.items():
    vmin = f"{np.nanmin(data):.2f}"
    vmax = f"{np.nanmax(data):.2f}"
    print(f"{name:<20} {str(data.shape):<20} {str(data.dtype):<10} {vmin:<10} {vmax:<10}")

# Store the shared profile for later use
ref_profile = list(rasters.values())[0][1]
print(f"\nCRS: {ref_profile['crs']}")
print(f"Resolution: {ref_profile['transform'][0]}m")

## 3. Visualization 1 — Individual Input Layer Maps

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(24, 12))
fig.suptitle('GEE Input Layers for Alaska Friction Surface', fontsize=16, y=1.02)

# --- LULC (categorical) ---
ax = axes[0, 0]
lulc_data = rasters['lulc'][0]
lulc_cmap = mcolors.ListedColormap([
    '#0077be',  # 0: water
    '#228b22',  # 1: trees
    '#90ee90',  # 2: grass
    '#006400',  # 3: flooded veg
    '#ffd700',  # 4: crops
    '#d2691e',  # 5: shrub
    '#ff4500',  # 6: built
    '#f5deb3',  # 7: bare
    '#ffffff',  # 8: snow/ice
])
im = ax.imshow(lulc_data, cmap=lulc_cmap, vmin=0, vmax=8)
ax.set_title('LULC (Dynamic World)')
ax.axis('off')
lulc_labels = ['Water', 'Trees', 'Grass', 'Flooded Veg', 'Crops',
               'Shrub', 'Built', 'Bare', 'Snow/Ice']
patches = [Patch(color=lulc_cmap(i/8), label=l) for i, l in enumerate(lulc_labels)]
ax.legend(handles=patches, loc='lower left', fontsize=6, ncol=2)

# --- Slope ---
ax = axes[0, 1]
slope_data = rasters['slope'][0]
im = ax.imshow(slope_data, cmap='YlOrRd', vmin=0, vmax=30)
ax.set_title('Slope (degrees)')
ax.axis('off')
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

# --- DEM ---
ax = axes[0, 2]
dem_data = rasters['dem'][0]
im = ax.imshow(dem_data, cmap='terrain', vmin=0, vmax=3000)
ax.set_title('Elevation (m)')
ax.axis('off')
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

# --- Permafrost ---
ax = axes[0, 3]
pf_data = rasters['permafrost'][0]
pf_cmap = mcolors.ListedColormap(['#f0f0f0', '#ffffb2', '#fd8d3c', '#bd0026'])
im = ax.imshow(pf_data, cmap=pf_cmap, vmin=0, vmax=3)
ax.set_title('Permafrost Zones')
ax.axis('off')
pf_labels = ['None', 'Sporadic', 'Discontinuous', 'Continuous']
patches = [Patch(color=pf_cmap(i/3), label=l) for i, l in enumerate(pf_labels)]
ax.legend(handles=patches, loc='lower left', fontsize=7)

# --- Roads Presence (GRIP4) ---
ax = axes[1, 0]
roads_data = rasters['roads_presence'][0]
im = ax.imshow(roads_data, cmap='binary_r', vmin=0, vmax=1)
ax.set_title('Roads (GRIP4 Presence)')
ax.axis('off')

# --- Rivers ---
ax = axes[1, 1]
rivers_data = rasters['rivers'][0]
riv_cmap = mcolors.ListedColormap(['#f0f0f0', '#0000ff', '#87ceeb'])
im = ax.imshow(rivers_data, cmap=riv_cmap, vmin=0, vmax=2)
ax.set_title('Rivers (NHD)')
ax.axis('off')
patches = [Patch(color=riv_cmap(i/2), label=l)
           for i, l in enumerate(['None', 'Major', 'Minor'])]
ax.legend(handles=patches, loc='lower left', fontsize=7)

# --- Empty / Info panel ---
for ax in [axes[1, 2], axes[1, 3]]:
    ax.axis('off')
axes[1, 2].text(0.5, 0.5, 'All layers: EPSG:3413\n150m resolution\nClipped to Alaska',
        transform=axes[1, 2].transAxes, ha='center', va='center', fontsize=12,
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.savefig('input_layers.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: input_layers.png")

## 4. Build Friction Surfaces

In [ ]:
# Build the three friction rasters
friction_road = friction_surface.build_friction_road(rasters)
friction_barge = friction_surface.build_friction_barge(rasters)
friction_sky = friction_surface.build_friction_sky(rasters)

# Save to disk
friction_surface.save_friction_rasters(friction_road, friction_barge, friction_sky, ref_profile)

# Print statistics
for name, arr in [('Road', friction_road), ('Barge', friction_barge), ('Sky', friction_sky)]:
    valid = arr[arr < friction_config.IMPASSABLE]
    impassable_pct = np.sum(arr >= friction_config.IMPASSABLE) / arr.size * 100
    print(f"\n{name} Friction Surface:")
    print(f"  Valid cells: min={valid.min():.2f}, mean={valid.mean():.2f}, max={valid.max():.2f}")
    print(f"  Impassable (999): {impassable_pct:.1f}% of cells")

## 5. Visualization 2 — Composite Friction Raster Maps

In [ ]:
# Custom colormap: green (low friction) -> yellow -> red (high), black = impassable
def friction_cmap():
    """Create a colormap for friction surfaces with black for impassable."""
    colors_list = ['#2d6a4f', '#52b788', '#b7e4c7', '#ffd166', '#ef476f', '#d00000']
    base_cmap = mcolors.LinearSegmentedColormap.from_list('friction', colors_list, N=256)
    base_cmap.set_over('black')  # 999 values shown as black
    return base_cmap

fcmap = friction_cmap()

fig, axes = plt.subplots(1, 3, figsize=(24, 8))
fig.suptitle('Composite Friction Surfaces', fontsize=16)

for ax, (name, arr) in zip(axes, [('Road', friction_road), ('Barge', friction_barge), ('Sky (Plane)', friction_sky)]):
    # Mask impassable for display
    display = np.where(arr >= friction_config.IMPASSABLE, np.nan, arr)
    im = ax.imshow(display, cmap=fcmap, vmin=1.0, vmax=5.0)

    # Overlay impassable as black
    impassable_mask = arr >= friction_config.IMPASSABLE
    ax.imshow(np.where(impassable_mask, 0.0, np.nan), cmap='gray', vmin=0, vmax=1, alpha=0.8)

    ax.set_title(f'{name} Friction', fontsize=14)
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='Friction Value',
                 extend='max')

# Add legend for black = impassable
fig.text(0.5, 0.02, 'Black = Impassable (999) | Green = Low Friction (~1.0) | Red = High Friction (~5.0)',
         ha='center', fontsize=11, style='italic')

plt.tight_layout()
plt.savefig('friction_surfaces.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: friction_surfaces.png")

## 6. Visualization 3 — Facility Overlay

In [ ]:
# Load facilities from vector file or DuckDB
vector_dir = os.getenv('VECTOR_DIR', '../vectors')
facilities_path = os.path.join(vector_dir, 'facilities_alaska.geojson')

if os.path.exists(facilities_path):
    facilities = gpd.read_file(facilities_path)
else:
    # Fall back to DuckDB
    con = duckdb.connect('../regionalization.duckdb', read_only=True)
    fac_df = con.execute("""
        SELECT f.facility_id, f.longitude, f.latitude, f.x_3413, f.y_3413,
               COALESCE(um.method_name, 'Unknown') AS delivery_method
        FROM facilities f
        LEFT JOIN uses_method um ON f.facility_id = um.facility_id
    """).fetchdf()
    con.close()
    facilities = gpd.GeoDataFrame(
        fac_df,
        geometry=gpd.points_from_xy(fac_df.x_3413, fac_df.y_3413),
        crs='EPSG:3413'
    )

# Method colors
method_colors = {'Road': '#2196F3', 'Barge': '#4CAF50', 'Plane': '#FF9800',
                 'Plane or Road': '#9C27B0', 'Unknown': '#757575'}

fig, ax = plt.subplots(1, 1, figsize=(16, 12))

# Background: road friction
display = np.where(friction_road >= friction_config.IMPASSABLE, np.nan, friction_road)
ax.imshow(display, cmap=fcmap, vmin=1.0, vmax=5.0, alpha=0.6)
ax.imshow(np.where(friction_road >= friction_config.IMPASSABLE, 0.0, np.nan),
          cmap='gray', vmin=0, vmax=1, alpha=0.4)

# Overlay facilities
transform = ref_profile['transform']
method_col = 'delivery_method' if 'delivery_method' in facilities.columns else 'Delivery_method'
for method, group in facilities.groupby(method_col):
    color = method_colors.get(method, '#757575')
    # Convert projected coordinates to pixel coordinates
    for _, row in group.iterrows():
        if hasattr(row.geometry, 'x'):
            col_px = int((row.geometry.x - transform.c) / transform.a)
            row_px = int((row.geometry.y - transform.f) / transform.e)
            ax.plot(col_px, row_px, 'o', color=color, markersize=4, markeredgecolor='white',
                    markeredgewidth=0.5)

# Legend
legend_patches = [Patch(color=c, label=m) for m, c in method_colors.items()
                  if m in facilities[method_col].unique()]
ax.legend(handles=legend_patches, loc='upper right', fontsize=10, title='Delivery Method')
ax.set_title('Bulk Fuel Facilities on Road Friction Surface', fontsize=14)
ax.axis('off')

plt.tight_layout()
plt.savefig('facility_overlay.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: facility_overlay.png")

## 7. Compute Least-Cost Paths

In [ ]:
# Run the full friction surface computation pipeline
# This computes least-cost paths for all connects_to edges
con = duckdb.connect('../regionalization.duckdb', read_only=False)

# Reproject facilities to EPSG:3413 (needed for raster lookups)
pipeline.reproject_facilities(con)

# Compute paths for each delivery method
for method, tif_name in [('Road', 'friction_road.tif'),
                          ('Barge', 'friction_barge.tif'),
                          ('Plane', 'friction_sky.tif')]:
    raster_dir = pipeline.get_raster_dir()
    tif_path = os.path.join(raster_dir, tif_name)
    if os.path.exists(tif_path):
        print(f"\nComputing least-cost paths for {method}...")
        results = friction_surface.compute_paths_for_method(con, tif_path, method)
        friction_surface.update_graph_friction(con, results)
        print(f"  Updated {len(results)} edges")
    else:
        print(f"  Skipping {method}: {tif_path} not found")

# Summary
stats = con.execute("""
    SELECT COUNT(*) AS total,
           COUNT(avg_friction) AS with_friction,
           ROUND(AVG(avg_friction), 3) AS mean_friction,
           ROUND(AVG(path_length_miles), 1) AS mean_path_mi
    FROM connects_to
""").fetchdf()
print(f"\nGraph summary:\n{stats.to_string(index=False)}")

con.close()

## 8. Visualization 4 — Least-Cost Path Traces

Select example facility pairs and plot the cost_pathway traces on the friction surface.

In [ ]:
# Query example edges with friction data from graph
con = duckdb.connect('../regionalization.duckdb', read_only=True)

examples = con.execute("""
    SELECT ct.src, ct.dst, ct.avg_friction, ct.path_length_miles, ct.distance_miles,
           f1.x_3413 AS src_x, f1.y_3413 AS src_y,
           f2.x_3413 AS dst_x, f2.y_3413 AS dst_y,
           COALESCE(um.method_name, 'Road') AS method
    FROM connects_to ct
    JOIN facilities f1 ON ct.src = f1.facility_id
    JOIN facilities f2 ON ct.dst = f2.facility_id
    LEFT JOIN uses_method um ON ct.src = um.facility_id
    WHERE ct.avg_friction IS NOT NULL
      AND ct.path_length_miles > 10
    ORDER BY ct.avg_friction DESC
    LIMIT 6
""").fetchdf()

con.close()

if len(examples) > 0:
    fig, axes = plt.subplots(2, 3, figsize=(24, 16))
    fig.suptitle('Least-Cost Path Traces on Friction Surfaces', fontsize=16)

    for idx, (_, row) in enumerate(examples.iterrows()):
        ax = axes[idx // 3, idx % 3]

        # Select appropriate friction surface
        if row['method'] == 'Barge':
            bg = friction_barge
        elif row['method'] == 'Plane':
            bg = friction_sky
        else:
            bg = friction_road

        display = np.where(bg >= friction_config.IMPASSABLE, np.nan, bg)
        ax.imshow(display, cmap=fcmap, vmin=1.0, vmax=5.0, alpha=0.7)

        # Plot source and destination as stars
        t = ref_profile['transform']
        src_col = int((row['src_x'] - t.c) / t.a)
        src_row = int((row['src_y'] - t.f) / t.e)
        dst_col = int((row['dst_x'] - t.c) / t.a)
        dst_row = int((row['dst_y'] - t.f) / t.e)

        ax.plot(src_col, src_row, '*', color='lime', markersize=15, markeredgecolor='black')
        ax.plot(dst_col, dst_row, '*', color='red', markersize=15, markeredgecolor='black')

        # Draw straight line for reference
        ax.plot([src_col, dst_col], [src_row, dst_row], '--', color='white', alpha=0.5, linewidth=1)

        detour = row['path_length_miles'] / row['distance_miles'] if row['distance_miles'] > 0 else 0
        ax.set_title(f"{row['method']}: {row['src']}->{row['dst']}\n"
                     f"Friction={row['avg_friction']:.2f}, "
                     f"Path={row['path_length_miles']:.0f}mi, "
                     f"Detour={detour:.1f}x", fontsize=10)
        ax.axis('off')

    plt.tight_layout()
    plt.savefig('path_traces.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Figure saved: path_traces.png")
else:
    print("No edges with friction data found. Run the computation step first.")

## 9. Visualization 5 — Haversine vs Friction Path Comparison

In [ ]:
# Compare Haversine straight-line vs friction least-cost path
con = duckdb.connect('../regionalization.duckdb', read_only=True)

compare = con.execute("""
    SELECT ct.src, ct.dst, ct.distance_miles AS haversine_mi,
           ct.path_length_miles AS friction_mi, ct.avg_friction,
           ROUND(ct.path_length_miles / NULLIF(ct.distance_miles, 0), 2) AS detour_ratio,
           COALESCE(um.method_name, 'Road') AS method
    FROM connects_to ct
    LEFT JOIN uses_method um ON ct.src = um.facility_id
    WHERE ct.path_length_miles IS NOT NULL
      AND ct.distance_miles > 5
    ORDER BY ct.path_length_miles / NULLIF(ct.distance_miles, 0) DESC
    LIMIT 100
""").fetchdf()
con.close()

if len(compare) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    fig.suptitle('Haversine vs Friction Path Comparison', fontsize=16)

    # Scatter: Haversine vs Friction distance
    ax = axes[0]
    for method, color in method_colors.items():
        subset = compare[compare['method'] == method]
        if len(subset) > 0:
            ax.scatter(subset['haversine_mi'], subset['friction_mi'],
                      c=color, label=method, alpha=0.6, s=20)
    max_val = max(compare['haversine_mi'].max(), compare['friction_mi'].max()) * 1.1
    ax.plot([0, max_val], [0, max_val], 'k--', alpha=0.3, label='1:1 line')
    ax.set_xlabel('Haversine Distance (miles)')
    ax.set_ylabel('Friction Path Distance (miles)')
    ax.set_title('Distance Comparison')
    ax.legend(fontsize=8)

    # Histogram of detour ratios
    ax = axes[1]
    for method, color in method_colors.items():
        subset = compare[compare['method'] == method]
        if len(subset) > 0:
            ax.hist(subset['detour_ratio'].dropna(), bins=20, color=color,
                   alpha=0.5, label=method)
    ax.set_xlabel('Detour Ratio (friction / haversine)')
    ax.set_ylabel('Count')
    ax.set_title('Path Detour Ratios')
    ax.axvline(x=1.0, color='black', linestyle='--', alpha=0.5)
    ax.legend(fontsize=8)

    # Box plot of detour ratio by method
    ax = axes[2]
    methods_present = compare['method'].unique()
    data_by_method = [compare[compare['method'] == m]['detour_ratio'].dropna()
                      for m in methods_present]
    bp = ax.boxplot(data_by_method, labels=methods_present, patch_artist=True)
    for patch, method in zip(bp['boxes'], methods_present):
        patch.set_facecolor(method_colors.get(method, '#757575'))
        patch.set_alpha(0.6)
    ax.set_ylabel('Detour Ratio')
    ax.set_title('Detour Ratio by Method')
    ax.axhline(y=1.0, color='black', linestyle='--', alpha=0.3)

    plt.tight_layout()
    plt.savefig('haversine_vs_friction.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Figure saved: haversine_vs_friction.png")
else:
    print("No comparison data available. Run friction computation first.")

## 10. Visualization 6 — Seasonal Friction Heatmaps

In [ ]:
# Seasonal friction comparison — focus on barge routes where variation is greatest
con = duckdb.connect('../regionalization.duckdb', read_only=True)

seasonal = con.execute("""
    SELECT li.region_name AS region,
           COALESCE(um.method_name, 'Road') AS method,
           ct.friction_summer, ct.friction_shoulder, ct.friction_winter,
           f1.x_3413 AS src_x, f1.y_3413 AS src_y
    FROM connects_to ct
    JOIN facilities f1 ON ct.src = f1.facility_id
    JOIN located_in li ON f1.facility_id = li.facility_id
    LEFT JOIN uses_method um ON f1.facility_id = um.facility_id
    WHERE ct.friction_summer IS NOT NULL
""").fetchdf()
con.close()

if len(seasonal) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(22, 7))
    fig.suptitle('Seasonal Friction Variation (Barge Routes)', fontsize=16)

    barge = seasonal[seasonal['method'] == 'Barge']
    if len(barge) == 0:
        barge = seasonal  # show all if no barge data

    for ax, (season, col) in zip(axes, [('Summer', 'friction_summer'),
                                         ('Shoulder (May/Oct)', 'friction_shoulder'),
                                         ('Winter', 'friction_winter')]):
        valid = barge[barge[col] < friction_config.IMPASSABLE]
        impassable = barge[barge[col] >= friction_config.IMPASSABLE]

        if len(valid) > 0:
            sc = ax.scatter(valid['src_x'], valid['src_y'], c=valid[col],
                          cmap='RdYlGn_r', vmin=1, vmax=10, s=15, alpha=0.7)
            plt.colorbar(sc, ax=ax, label='Friction')

        if len(impassable) > 0:
            ax.scatter(impassable['src_x'], impassable['src_y'],
                      c='black', s=15, alpha=0.7, marker='x', label='Impassable')

        ax.set_title(f'{season}\n(valid: {len(valid)}, impassable: {len(impassable)})')
        ax.set_aspect('equal')
        ax.tick_params(labelsize=7)
        if len(impassable) > 0:
            ax.legend(fontsize=8)

    plt.tight_layout()
    plt.savefig('seasonal_friction.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Figure saved: seasonal_friction.png")
else:
    print("No seasonal data. Run friction agents first.")

## 11. Visualization 7 — Regional Friction Summary

In [ ]:
# Regional friction summary — grouped bar chart
con = duckdb.connect('../regionalization.duckdb', read_only=True)

regional = con.execute("""
    SELECT li.region_name AS region,
           COALESCE(um.method_name, 'Unknown') AS method,
           AVG(ct.avg_friction) AS mean_friction,
           MIN(ct.avg_friction) AS min_friction,
           MAX(ct.avg_friction) AS max_friction,
           COUNT(*) AS edge_count
    FROM connects_to ct
    JOIN facilities f ON ct.src = f.facility_id
    JOIN located_in li ON f.facility_id = li.facility_id
    LEFT JOIN uses_method um ON f.facility_id = um.facility_id
    WHERE ct.avg_friction IS NOT NULL AND ct.avg_friction < 999
    GROUP BY li.region_name, um.method_name
    ORDER BY li.region_name, um.method_name
""").fetchdf()
con.close()

if len(regional) > 0:
    regions = regional['region'].unique()
    methods = regional['method'].unique()

    fig, ax = plt.subplots(figsize=(16, 8))

    x = np.arange(len(regions))
    width = 0.8 / len(methods)

    for i, method in enumerate(methods):
        subset = regional[regional['method'] == method]
        # Align with region index
        means = [subset[subset['region'] == r]['mean_friction'].values[0]
                 if r in subset['region'].values else 0 for r in regions]
        mins = [subset[subset['region'] == r]['min_friction'].values[0]
                if r in subset['region'].values else 0 for r in regions]
        maxs = [subset[subset['region'] == r]['max_friction'].values[0]
                if r in subset['region'].values else 0 for r in regions]

        yerr_low = [m - mn for m, mn in zip(means, mins)]
        yerr_high = [mx - m for m, mx in zip(means, maxs)]

        color = method_colors.get(method, '#757575')
        ax.bar(x + i * width, means, width, label=method, color=color, alpha=0.7,
               yerr=[yerr_low, yerr_high], capsize=3)

    ax.set_xlabel('Region', fontsize=12)
    ax.set_ylabel('Average Friction', fontsize=12)
    ax.set_title('Average Friction by Region and Delivery Method', fontsize=14)
    ax.set_xticks(x + width * (len(methods) - 1) / 2)
    ax.set_xticklabels(regions, rotation=45, ha='right', fontsize=9)
    ax.legend(title='Delivery Method')
    ax.axhline(y=1.0, color='gray', linestyle='--', alpha=0.3, label='Baseline (1.0)')

    plt.tight_layout()
    plt.savefig('regional_friction_summary.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Figure saved: regional_friction_summary.png")
else:
    print("No regional friction data available.")

## 12. Visualization 8 — Cost Distribution Histograms

In [ ]:
# Delivery cost distributions by method
con = duckdb.connect('../regionalization.duckdb', read_only=True)

costs = con.execute("""
    SELECT ct.delivery_cost, ct.cost_summer, ct.cost_shoulder, ct.cost_winter,
           COALESCE(um.method_name, 'Road') AS method
    FROM connects_to ct
    LEFT JOIN uses_method um ON ct.src = um.facility_id
    WHERE ct.delivery_cost IS NOT NULL
""").fetchdf()
con.close()

if len(costs) > 0:
    methods = costs['method'].unique()
    fig, axes = plt.subplots(1, len(methods), figsize=(7 * len(methods), 6))
    if len(methods) == 1:
        axes = [axes]

    fig.suptitle('Delivery Cost Distribution by Method', fontsize=16)

    # ISER/AEA benchmark ranges ($/mi approximation)
    benchmarks = {
        'Road': (2.0, 5.0),
        'Barge': (1.0, 3.0),
        'Plane': (8.0, 15.0),
    }

    for ax, method in zip(axes, methods):
        subset = costs[costs['method'] == method]['delivery_cost'].dropna()
        if len(subset) > 0:
            color = method_colors.get(method, '#757575')
            ax.hist(subset, bins=30, color=color, alpha=0.7, edgecolor='white')

            # Add benchmark range
            bm = benchmarks.get(method)
            if bm:
                ax.axvline(x=subset.median(), color='black', linestyle='-',
                          linewidth=2, label=f'Median: ${subset.median():.0f}')

            ax.set_xlabel('Delivery Cost ($)')
            ax.set_ylabel('Count')
            ax.set_title(f'{method}\n(n={len(subset)}, median=${subset.median():.0f})')
            ax.legend(fontsize=8)

    plt.tight_layout()
    plt.savefig('cost_distributions.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Figure saved: cost_distributions.png")
else:
    print("No cost data. Run friction agents first.")

## 13. Summary Statistics

In [ ]:
# Final summary statistics from the graph database
con = duckdb.connect('../regionalization.duckdb', read_only=True)

print("=" * 70)
print("FRICTION SURFACE SUMMARY")
print("=" * 70)

# Overall stats
overall = con.execute("""
    SELECT
        COUNT(*) AS total_edges,
        COUNT(avg_friction) AS with_friction,
        COUNT(delivery_cost) AS with_cost,
        COUNT(friction_winter) AS with_seasonal
    FROM connects_to
""").fetchdf()
print(f"\nEdge Coverage:")
print(f"  Total connects_to edges: {overall['total_edges'].values[0]}")
print(f"  With friction data:      {overall['with_friction'].values[0]}")
print(f"  With seasonal data:      {overall['with_seasonal'].values[0]}")
print(f"  With delivery cost:      {overall['with_cost'].values[0]}")

# By method
by_method = con.execute("""
    SELECT
        COALESCE(um.method_name, 'Unknown') AS method,
        COUNT(*) AS edges,
        ROUND(AVG(ct.avg_friction), 3) AS mean_friction,
        ROUND(MEDIAN(ct.avg_friction), 3) AS median_friction,
        ROUND(MAX(ct.avg_friction), 3) AS max_friction,
        ROUND(AVG(ct.delivery_cost), 0) AS mean_cost,
        ROUND(MEDIAN(ct.delivery_cost), 0) AS median_cost,
        ROUND(AVG(ct.path_length_miles), 1) AS mean_path_mi,
        ROUND(AVG(ct.path_length_miles / NULLIF(ct.distance_miles, 0)), 2) AS mean_detour
    FROM connects_to ct
    LEFT JOIN uses_method um ON ct.src = um.facility_id
    WHERE ct.avg_friction IS NOT NULL
    GROUP BY um.method_name
    ORDER BY um.method_name
""").fetchdf()
print(f"\nFriction & Cost by Delivery Method:")
print(by_method.to_string(index=False))

# Winter impassability
winter = con.execute("""
    SELECT
        COALESCE(um.method_name, 'Unknown') AS method,
        COUNT(*) AS total,
        SUM(CASE WHEN ct.friction_winter >= 999 THEN 1 ELSE 0 END) AS impassable_winter,
        ROUND(100.0 * SUM(CASE WHEN ct.friction_winter >= 999 THEN 1 ELSE 0 END) / COUNT(*), 1)
            AS pct_impassable
    FROM connects_to ct
    LEFT JOIN uses_method um ON ct.src = um.facility_id
    WHERE ct.friction_winter IS NOT NULL
    GROUP BY um.method_name
""").fetchdf()
print(f"\nWinter Impassability:")
print(winter.to_string(index=False))

con.close()
print("\n" + "=" * 70)
print("Done.")